In [2]:
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import os

# 파일 경로 설정
INPUT_PATH = '/Users/judetak/Desktop/산공카르텔경진대회/생활인구 데이터_격자 병합 완료.csv'
OUTPUT_PATH = '/Users/judetak/Desktop/산공카르텔경진대회/생활인구 데이터_격자 병합 완료.parquet'

# 한 번에 메모리에 올릴 행의 수 (200만 행 정도면 일반적인 노트북 RAM에서도 충분히 버팁니다)
CHUNK_SIZE = 2000000 

print("🚀 대용량 CSV -> Parquet 변환 시작...")

writer = None
total_rows = 0

# chunksize를 지정하여 파일을 조각내어 읽어옵니다. (low_memory=False로 타입 추론 경고 방지)
for i, chunk in enumerate(pd.read_csv(INPUT_PATH, chunksize=CHUNK_SIZE, low_memory=False)):
    
    # ---------------------------------------------------------
    # [핵심 최적화] 64비트 자료형을 32비트로 다운캐스팅하여 용량 절반으로 줄이기
    # ---------------------------------------------------------
    float_cols = chunk.select_dtypes(include=['float64']).columns
    chunk[float_cols] = chunk[float_cols].astype('float32')
    
    int_cols = chunk.select_dtypes(include=['int64']).columns
    chunk[int_cols] = chunk[int_cols].astype('int32')

    # Pandas DataFrame을 PyArrow Table로 변환
    table = pa.Table.from_pandas(chunk)

    # 첫 번째 청크(i==0)에서 데이터 스키마를 파악하고 파일 쓰기(Writer) 객체 생성
    if i == 0:
        writer = pq.ParquetWriter(OUTPUT_PATH, table.schema, compression='snappy')
    
    # Parquet 파일에 현재 청크 데이터를 추가 (Append)
    writer.write_table(table)
    
    total_rows += len(chunk)
    print(f"  ✅ {(i * CHUNK_SIZE):,} ~ {total_rows:,}행 처리 및 병합 완료")

# 모든 작업이 끝나면 Writer를 닫아 파일을 완성합니다.
if writer:
    writer.close()

# 변환 전/후 파일 크기 비교
csv_size = os.path.getsize(INPUT_PATH) / (1024 ** 3)
pq_size = os.path.getsize(OUTPUT_PATH) / (1024 ** 3)

print("-" * 50)
print(f"🎉 변환 완료!")
print(f"📂 저장 경로: {OUTPUT_PATH}")
print(f"📊 총 처리된 데이터: {total_rows:,}행")
print(f"💾 용량 변화: {csv_size:.2f} GB (CSV) ➡️ {pq_size:.2f} GB (Parquet)")

🚀 대용량 CSV -> Parquet 변환 시작...
  ✅ 0 ~ 2,000,000행 처리 및 병합 완료
  ✅ 2,000,000 ~ 4,000,000행 처리 및 병합 완료
  ✅ 4,000,000 ~ 6,000,000행 처리 및 병합 완료
  ✅ 6,000,000 ~ 8,000,000행 처리 및 병합 완료
  ✅ 8,000,000 ~ 10,000,000행 처리 및 병합 완료
  ✅ 10,000,000 ~ 12,000,000행 처리 및 병합 완료
  ✅ 12,000,000 ~ 14,000,000행 처리 및 병합 완료
  ✅ 14,000,000 ~ 16,000,000행 처리 및 병합 완료
  ✅ 16,000,000 ~ 18,000,000행 처리 및 병합 완료
  ✅ 18,000,000 ~ 20,000,000행 처리 및 병합 완료
  ✅ 20,000,000 ~ 22,000,000행 처리 및 병합 완료
  ✅ 22,000,000 ~ 24,000,000행 처리 및 병합 완료
  ✅ 24,000,000 ~ 26,000,000행 처리 및 병합 완료
  ✅ 26,000,000 ~ 28,000,000행 처리 및 병합 완료
  ✅ 28,000,000 ~ 30,000,000행 처리 및 병합 완료
  ✅ 30,000,000 ~ 32,000,000행 처리 및 병합 완료
  ✅ 32,000,000 ~ 34,000,000행 처리 및 병합 완료
  ✅ 34,000,000 ~ 36,000,000행 처리 및 병합 완료
  ✅ 36,000,000 ~ 38,000,000행 처리 및 병합 완료
  ✅ 38,000,000 ~ 40,000,000행 처리 및 병합 완료
  ✅ 40,000,000 ~ 42,000,000행 처리 및 병합 완료
  ✅ 42,000,000 ~ 44,000,000행 처리 및 병합 완료
  ✅ 44,000,000 ~ 46,000,000행 처리 및 병합 완료
  ✅ 46,000,000 ~ 48,000,000행 처리 및 병합 완료
  ✅ 48,000,000 ~ 50,000,000